# 5장 Layout Out Pages

## 5.1 The Layout Tree

지금까지는 `Layout` 클래스가 HTML DOM Tree를 순회하며 화면에 그릴 모든 텍스트의 좌표를 계산했다.

하지만 실제로는 요소마다 레이아웃 방식(블록, 인라인, 플렉스 등)이 다르기 때문에, 레이아웃 트리를 분리한다.



지금까지(Chapter 4)는 단일 `Layout` 클래스가 HTML 트리(DOM)를 순회하며 화면에 그릴 모든 텍스트의 좌표를 계산했습니다. 
하지만 실제 브라우저는 요소마다 레이아웃 방식(블록, 인라인, 플렉스 등)이 다르기 때문에, 이를 하나의 클래스에서 모두 처리하면 코드가 지나치게 복잡해집니다.

이를 해결하기 위해 5장에서는 **HTML 트리(DOM Tree)와 레이아웃 트리(Layout Tree)를 분리**합니다. 

#### 1. 레이아웃 트리의 핵심 개념
- HTML 트리의 노드가 "문서의 구조(의미)"를 나타낸다면, 레이아웃 트리의 노드는 화면에 그려지는 **"시각적인 박스(Box)"** 를 나타냅니다.
- 각 레이아웃 노드는 자신만의 `x`, `y`, `width`, `height`를 가집니다.
- 각 노드는 자신의 자식 노드들에게 "너비(width)"를 알려주고, 자식들은 레이아웃을 마친 뒤 자신의 "높이(height)"를 부모에게 반환하는 재귀적인 구조를 가집니다.

#### 2. 주요 클래스 리팩토링
- **`DocumentLayout`**: 레이아웃 트리의 최상위 루트 노드입니다. 브라우저 창 전체를 나타냅니다.
- **`BlockLayout`**: `<p>`, `<h1>`, `<div>` 등 위에서 아래로 쌓이는 **블록 요소**를 담당하는 레이아웃 노드입니다. 이전 부모(`parent`)와 이전 형제(`previous`) 노드의 참조를 가지고 있어 자신의 Y 좌표를 계산하는 데 사용합니다.

#### 3. 변경된 렌더링 흐름
이전에는 `Layout(nodes).display_list`로 한 번에 처리했지만, 이제는 각 레이아웃 클래스에 있는 `layout()` 메서드를 호출하여 계층적으로 레이아웃을 계산합니다.
```python
# browser.py 변경점 요약
self.document = DocumentLayout(self.nodes)
self.document.layout()
self.display_list = self.document.display_list
```